# 03 — Compare Transformer and TITANS
Separates within-genome continuation from ANI-held-out generalization, checks the controlled training contract, and bootstraps paired stream-level differences. One seed remains exploratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/ATCG-FM || git clone https://github.com/DRAGGON-Lab/ATCG-FM.git /content/ATCG-FM
%pip install -q pandas matplotlib numpy -e /content/ATCG-FM/packages/atcg-sequence --no-deps -e /content/ATCG-FM/packages/atcg-models --no-deps -e /content/ATCG-FM/packages/atcg-runtime --no-deps


In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from atcg.models import GenomicLanguageModel, attention_tiny, titans_memory_tiny
from atcg.runtime import ComparisonCandidate, ComparisonInvariants, ComparisonPlan
from atcg.sequence import FixedAlphabetTokenizer
ROOT = Path('/content/drive/MyDrive/ATCG-FM/ecoli_hybrid_v1'); STAGE_NAME = 'stage-small'; OUT = ROOT / 'comparison' / STAGE_NAME; OUT.mkdir(parents=True, exist_ok=True)
attention = json.loads((ROOT / f'runs/{STAGE_NAME}/attention-seed17/result.json').read_text())
titans = json.loads((ROOT / f'runs/{STAGE_NAME}/titans-memory-seed17/result.json').read_text())
if attention['experiment'] != titans['experiment']: raise RuntimeError('candidate experiment configurations differ')
experiment = attention['experiment']; profile = experiment['profile']; tokenizer = FixedAlphabetTokenizer()
if attention['training']['tokens'] != titans['training']['tokens'] or attention['training']['tokens'] != experiment['train_tokens']: raise RuntimeError('training token contract differs')
attention_config = attention_tiny(tokenizer.vocab_size, max_seq_len=128, d_model=profile['d_model'], n_layers=profile['layers'], n_heads=profile['heads'])
titans_config = titans_memory_tiny(tokenizer.vocab_size, max_seq_len=128, d_model=profile['d_model'], n_layers=profile['layers'], expansion_factor=2, projection_kernel_size=4)
plan = ComparisonPlan(comparison_id=experiment['comparison_id'], substitution_unit='mixer', invariants=ComparisonInvariants(tokenizer_id='iupac-character-v1', dataset_split=experiment['dataset_fingerprint'], training_tokens=experiment['train_tokens'], optimizer_protocol='adamw-ordered-fp16-v1', segment_length=128, gradient_horizon=1), candidates=(ComparisonCandidate('attention', attention_config), ComparisonCandidate('titans-memory', titans_config)))
comparison_manifest = plan.manifest({'attention':GenomicLanguageModel(attention_config),'titans-memory':GenomicLanguageModel(titans_config)})
(OUT / 'comparison_manifest.json').write_text(json.dumps(comparison_manifest, indent=2, sort_keys=True) + '\n')


In [ ]:
rows = []
for split in experiment['evaluation_splits']:
    candidates = [('attention','standard',attention['evaluation'][split]),('titans-memory','adaptive',titans['evaluation']['adaptive'][split]),('titans-memory','reset_each_segment',titans['evaluation']['reset_each_segment'][split])]
    for candidate, policy, values in candidates:
        training = attention['training'] if candidate == 'attention' else titans['training']
        rows.append({'candidate':candidate,'state_policy':policy,'split':split,'bits_per_base':values['bits_per_token'],'perplexity':values['perplexity'],'token_accuracy':values['token_accuracy'],'tokens':values['token_count'],'parameters':attention['parameters'] if candidate == 'attention' else titans['parameters'],'train_tokens_per_second':training['tokens_per_second'],'wall_time_minutes':training['wall_time_seconds']/60,'peak_memory_gib':training['peak_memory_bytes']/1024**3})
summary = pd.DataFrame(rows); display(summary)
summary.to_csv(OUT / 'summary.csv', index=False)
for candidate, policy in [('attention','standard'),('titans-memory','adaptive'),('titans-memory','reset_each_segment')]:
    values = summary[(summary.candidate == candidate) & (summary.state_policy == policy)].set_index('split').bits_per_base
    print(candidate, policy, 'held-out-clade minus within-genome BpB:', values.test_clade - values.test_within)
adaptive = summary[(summary.candidate == 'titans-memory') & (summary.state_policy == 'adaptive')].set_index('split').bits_per_base
reset = summary[(summary.candidate == 'titans-memory') & (summary.state_policy == 'reset_each_segment')].set_index('split').bits_per_base
print('TITANS adaptive minus reset BpB:', (adaptive-reset).to_dict())


In [ ]:
def paired_bootstrap(split, repetitions=2000):
    left = pd.DataFrame(attention['stream_evaluation'][split]).set_index('stream_id')
    right = pd.DataFrame(titans['stream_evaluation'][split]).set_index('stream_id')
    joined = left.join(right, lsuffix='_attention', rsuffix='_titans', how='inner', validate='one_to_one')
    if len(joined) != len(left) or len(joined) != len(right): raise RuntimeError(f'{split}: stream pairing differs')
    delta = joined.bits_per_token_titans.to_numpy() - joined.bits_per_token_attention.to_numpy()
    weights = joined.token_count_attention.to_numpy(); rng = np.random.default_rng(17); estimates = []
    for _ in range(repetitions):
        indices = rng.integers(0, len(joined), len(joined)); estimates.append(np.average(delta[indices], weights=weights[indices]))
    return {'split':split,'streams':len(joined),'titans_minus_attention_bpb':float(np.average(delta,weights=weights)),'ci95_low':float(np.quantile(estimates,.025)),'ci95_high':float(np.quantile(estimates,.975))}
bootstrap = pd.DataFrame([paired_bootstrap(split) for split in ('test_within','test_clade')]); display(bootstrap)
bootstrap.to_csv(OUT / 'paired_bootstrap.csv', index=False)
offset_rows = []
for candidate, policy, result in [('attention','standard',attention),('titans-memory','adaptive',titans),('titans-memory','reset_each_segment',titans)]:
    for split in ('test_within','test_clade'):
        values = result['evaluation'][split] if candidate == 'attention' else result['evaluation'][policy][split]
        for offset, metrics in values['offset_bins'].items(): offset_rows.append({'candidate':candidate,'state_policy':policy,'split':split,'offset':offset,'bits_per_base':metrics['bits_per_token'],'tokens':metrics['token_count']})
offsets = pd.DataFrame(offset_rows); offsets.to_csv(OUT / 'offset_quality.csv', index=False); display(offsets)


In [ ]:
def losses(path): return pd.DataFrame([json.loads(line) for line in path.read_text().splitlines()])
attention_loss = losses(ROOT / f'runs/{STAGE_NAME}/attention-seed17/metrics.jsonl'); titans_loss = losses(ROOT / f'runs/{STAGE_NAME}/titans-memory-seed17/metrics.jsonl')
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(attention_loss.tokens_seen, attention_loss.loss, alpha=.7, label='Transformer'); axes[0].plot(titans_loss.tokens_seen, titans_loss.loss, alpha=.7, label='TITANS'); axes[0].set(xlabel='training tokens',ylabel='NLL',title='Identical training schedule'); axes[0].legend()
primary = summary[~((summary.candidate == 'titans-memory') & (summary.state_policy == 'reset_each_segment'))].copy(); primary['label'] = primary.candidate + '\n' + primary.split
primary.plot.bar(x='label',y='bits_per_base',ax=axes[1],legend=False,title='Within vs held-out quality',ylabel='bits/base')
for (candidate,policy,split),group in offsets.groupby(['candidate','state_policy','split']): axes[2].plot(group.offset,group.bits_per_base,marker='o',label=f'{candidate}/{policy}/{split}')
axes[2].set(xlabel='target offset in 16,384-base stream',ylabel='bits/base',title='Quality by stream offset'); axes[2].tick_params(axis='x',rotation=30); axes[2].legend(fontsize=6)
fig.tight_layout(); fig.savefig(OUT / 'comparison.png', dpi=160, bbox_inches='tight'); plt.show()


## Interpretation boundary
`test_within` measures unseen coordinates from genomes represented in training; `test_clade` measures transfer to ANI-99-separated genomes. Report both and their gap. The segment-reset TITANS result is an ablation, not a third trained model. Repeat with seeds 29 and 43 before making a comparative claim.